In [1]:
!pip install transformers datasets scikit-learn torch gradio

   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/527.0 kB ? eta -:--:-

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.37.1 requires rich<14,>=10.14.0, but you have rich 15.0.0 which is incompatible.


   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 13.9/27.4 MB 140.3 kB/s eta 0:01:36
   -------------------- ------------------- 14.2/27.4 MB 151.2 kB/s eta 0:01:28
   -------------------- ------------------- 14.2/27.4 MB 151.2 kB/s eta 0:01:28
   -------------------- ------------------- 14.2/27.4 MB 151.2 kB/s eta 0:01:28
   -------------------- ------------------- 14.2/27.4 MB 151.2 kB/s eta 0:01:28
   -------------------- ------------------- 14.2/27.4 MB 151.2 kB/s eta 0:01:28
   -------------------- ----------------

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

In [17]:
dataset = load_dataset("ag_news")

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [19]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [21]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [23]:
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [25]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    
    return {"accuracy": acc, "f1": f1}

In [29]:
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
test_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

In [31]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # keep 1 for speed
    weight_decay=0.01,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

NameError: name 'training_args' is not defined

In [35]:
trainer.train()

NameError: name 'trainer' is not defined

In [37]:
trainer.evaluate()

NameError: name 'trainer' is not defined

In [39]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class_id = logits.argmax().item()
    return predicted_class_id

print(predict("Apple releases new iPhone"))

0


In [41]:
import gradio as gr

labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(text):
    pred = predict(text)
    return labels[pred]

interface = gr.Interface(fn=classify_news, inputs="text", outputs="text")

interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [43]:
model.save_pretrained("bert-news-model")
tokenizer.save_pretrained("bert-news-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('bert-news-model\\tokenizer_config.json', 'bert-news-model\\tokenizer.json')